# Parking Birmingham — Feature Engineering and Temporal Splitting

**Course:** ADSP 32021 — Machine Learning Operations  
**Project:** Parking Availability Prediction Using an Automated MLOps Pipeline

## 1. Introduction

This notebook develops leakage-safe features for predicting parking occupancy approximately 30 minutes into the future using the validated UCI Parking Birmingham dataset.

### Objectives

1. Load and validate parking occupancy observations.
2. Construct future occupancy targets using a 25–35-minute forecasting window.
3. Engineer temporal, facility-specific, and historical occupancy features.
4. Prevent data leakage by ensuring features use only information available at prediction time.
5. Split observations chronologically into training, validation, and test sets.
6. Prepare reproducible datasets for baseline model development.

### Data Leakage Prevention

All input features must be available at or before the prediction timestamp.

Future occupancy values will be used exclusively as prediction targets, never as input features.

Chronological splitting will preserve the temporal ordering of observations, and examples whose target timestamps cross split boundaries will be excluded.


## 2. Data Loading and Validation

The raw Parking Birmingham dataset is loaded and processed using the reusable `validate_parking_data()` function developed during the initial data audit.

This function removes duplicate observations, excludes invalid occupancy measurements, converts timestamps to datetime format, and sorts observations chronologically within each parking facility.

Reusing the validation module ensures consistent preprocessing across exploratory analysis, model training, and the future automated Airflow pipeline.

The original dataset remains unchanged to preserve reproducibility.


In [15]:
import pandas as pd
import sys
from pathlib import Path

# Locate the project root from the notebooks directory
project_root = Path.cwd().resolve().parent

# Allow imports from the project's src directory
sys.path.insert(0, str(project_root))

from src.data_validation import validate_parking_data

# Load the original dataset
data_path = project_root / "data" / "dataset.csv"
df_raw = pd.read_csv(data_path)

# Apply the reusable validation function
df = validate_parking_data(df_raw)

# Verify the validated dataset
print("Original observations:", len(df_raw))
print("Validated observations:", len(df))
print("Parking facilities:", df["SystemCodeNumber"].nunique())

print("\nData types:")
print(df.dtypes)

display(df.head())


Original observations: 35717
Validated observations: 35116
Parking facilities: 30

Data types:
SystemCodeNumber            object
Capacity                     int64
Occupancy                    int64
LastUpdated         datetime64[ns]
dtype: object


,SystemCodeNumber,Capacity,Occupancy,LastUpdated
0,BHMBCCMKT01,577,61,2016-10-04 07:59:42
1,BHMBCCMKT01,577,64,2016-10-04 08:25:42
2,BHMBCCMKT01,577,80,2016-10-04 08:59:42
3,BHMBCCMKT01,577,107,2016-10-04 09:32:46
4,BHMBCCMKT01,577,150,2016-10-04 09:59:48



### 2.1 Validated Dataset Summary

The reusable validation pipeline successfully processed the original dataset.

- Original observations: 35,717
- Validated observations: 35,116
- Parking facilities retained: 30
- Timestamp format: datetime64[ns]

The validated dataset is sorted chronologically within each parking facility and contains no duplicate facility-timestamp records or invalid occupancy measurements.

## 3. Forecast Target Construction

The forecasting objective is to predict parking occupancy approximately 30 minutes after the current observation.

Because timestamps are irregular, we define an acceptable prediction horizon of **25–35 minutes**.

For each parking facility, the next recorded occupancy is selected as the target only when its timestamp falls within this interval.

Observations without an eligible future measurement are excluded from the supervised modeling dataset.

**Leakage prevention:** Future occupancy and target timestamps are used only for target construction and evaluation. They will not be included as model input features.

In [16]:
# Ensure observations are sorted within each facility
df = df.sort_values(
    ["SystemCodeNumber", "LastUpdated"]
).reset_index(drop=True)

# Identify the next observation within each facility
facility_groups = df.groupby("SystemCodeNumber")

df["TargetTimestamp"] = (
    facility_groups["LastUpdated"].shift(-1)
)

df["TargetOccupancy"] = (
    facility_groups["Occupancy"].shift(-1)
)

# Calculate the actual forecasting horizon
df["ForecastHorizonMinutes"] = (
    (df["TargetTimestamp"] - df["LastUpdated"])
    .dt.total_seconds() / 60
)

# Keep only observations with a 25–35 minute horizon
df_model = df.loc[
    df["ForecastHorizonMinutes"].between(25, 35)
].copy()

# Verify target construction
print("Validated observations:", len(df))
print("Valid forecasting examples:", len(df_model))

print("\nForecast horizon statistics (minutes):")
print(df_model["ForecastHorizonMinutes"].describe())

print("\nMissing target values:",
      df_model["TargetOccupancy"].isna().sum())

display(
    df_model[
        [
            "SystemCodeNumber",
            "LastUpdated",
            "Occupancy",
            "TargetTimestamp",
            "TargetOccupancy",
            "ForecastHorizonMinutes"
        ]
    ].head(10)
)


Validated observations: 35116
Valid forecasting examples: 31327

Forecast horizon statistics (minutes):
count    31327.000000
mean        29.988520
std          3.318462
min         25.916667
25%         26.966667
50%         29.983333
75%         33.033333
max         34.433333
Name: ForecastHorizonMinutes, dtype: float64

Missing target values: 0


,SystemCodeNumber,LastUpdated,Occupancy,TargetTimestamp,TargetOccupancy,ForecastHorizonMinutes
0,BHMBCCMKT01,2016-10-04 07:59:42,61,2016-10-04 08:25:42,64.0,26.000000
1,BHMBCCMKT01,2016-10-04 08:25:42,64,2016-10-04 08:59:42,80.0,34.000000
2,BHMBCCMKT01,2016-10-04 08:59:42,80,2016-10-04 09:32:46,107.0,33.066667
3,BHMBCCMKT01,2016-10-04 09:32:46,107,2016-10-04 09:59:48,150.0,27.033333
4,BHMBCCMKT01,2016-10-04 09:59:48,150,2016-10-04 10:26:49,177.0,27.016667
5,BHMBCCMKT01,2016-10-04 10:26:49,177,2016-10-04 10:59:48,219.0,32.983333
6,BHMBCCMKT01,2016-10-04 10:59:48,219,2016-10-04 11:25:47,247.0,25.983333
7,BHMBCCMKT01,2016-10-04 11:25:47,247,2016-10-04 11:59:44,259.0,33.950000
8,BHMBCCMKT01,2016-10-04 11:59:44,259,2016-10-04 12:29:45,266.0,30.016667
9,BHMBCCMKT01,2016-10-04 12:29:45,266,2016-10-04 13:02:48,269.0,33.050000


### 3.1 Forecast Target Construction Results

After validating and cleaning the Parking Birmingham dataset, **31,327 supervised forecasting examples** were generated from 35,116 validated observations.

Each example uses the current parking observation to predict occupancy at the next recorded timestamp, provided that the next observation occurs within 25–35 minutes.

**Forecast horizon statistics:**

| Metric | Result |
|---|---:|
| Valid forecasting examples | 31,327 |
| Percentage of validated observations retained | 89.21% |
| Mean forecast horizon | 29.99 minutes |
| Median forecast horizon | 29.98 minutes |
| Minimum forecast horizon | 25.92 minutes |
| Maximum forecast horizon | 34.43 minutes |
| Missing target values | 0 |

**Interpretation:**

The results support using an approximately 30-minute forecasting horizon for the project.

The timestamp-aware target construction prevents observations separated by long reporting gaps from being incorrectly treated as 30-minute forecasting examples.

Future occupancy is stored separately as `TargetOccupancy` and will be used only as the supervised learning label.

`TargetTimestamp` and `ForecastHorizonMinutes` will be excluded from model inputs because their values are derived from future observations and would not necessarily be known at prediction time.


## 4. Time-Based Feature Engineering

Parking occupancy may exhibit recurring temporal patterns associated with time of day, day of the week, and weekends.

We extract calendar features from `LastUpdated`, which represents the timestamp available when a prediction is requested.

The following features will be created:

- `hour`: Hour of the observation (0–23).
- `day_of_week`: Day of the week (Monday = 0, Sunday = 6).
- `is_weekend`: Binary indicator for Saturday or Sunday.
- `month`: Calendar month of the observation.

These features are derived exclusively from the current observation timestamp and do not require future information.

Their predictive usefulness will be evaluated during model training.


In [17]:
# Extract calendar features from the current observation timestamp
df_model["hour"] = df_model["LastUpdated"].dt.hour

df_model["day_of_week"] = (
    df_model["LastUpdated"].dt.dayofweek
)

df_model["is_weekend"] = (
    df_model["day_of_week"] >= 5
).astype(int)

df_model["month"] = df_model["LastUpdated"].dt.month

# Inspect engineered features
display(
    df_model[
        [
            "SystemCodeNumber",
            "LastUpdated",
            "hour",
            "day_of_week",
            "is_weekend",
            "month",
            "Occupancy",
            "TargetOccupancy"
        ]
    ].head(10)
)

# Verify missing values
print("\nMissing values in time-based features:")
print(
    df_model[
        ["hour", "day_of_week", "is_weekend", "month"]
    ].isna().sum()
)


,SystemCodeNumber,LastUpdated,hour,day_of_week,is_weekend,month,Occupancy,TargetOccupancy
0,BHMBCCMKT01,2016-10-04 07:59:42,7,1,0,10,61,64.0
1,BHMBCCMKT01,2016-10-04 08:25:42,8,1,0,10,64,80.0
2,BHMBCCMKT01,2016-10-04 08:59:42,8,1,0,10,80,107.0
3,BHMBCCMKT01,2016-10-04 09:32:46,9,1,0,10,107,150.0
4,BHMBCCMKT01,2016-10-04 09:59:48,9,1,0,10,150,177.0
5,BHMBCCMKT01,2016-10-04 10:26:49,10,1,0,10,177,219.0
6,BHMBCCMKT01,2016-10-04 10:59:48,10,1,0,10,219,247.0
7,BHMBCCMKT01,2016-10-04 11:25:47,11,1,0,10,247,259.0
8,BHMBCCMKT01,2016-10-04 11:59:44,11,1,0,10,259,266.0
9,BHMBCCMKT01,2016-10-04 12:29:45,12,1,0,10,266,269.0



Missing values in time-based features:
hour           0
day_of_week    0
is_weekend     0
month          0
dtype: int64


### 4.1 Time-Based Feature Engineering Results

Four temporal features were successfully generated from the current observation timestamp:

- `hour`: Captures hourly parking occupancy patterns.
- `day_of_week`: Represents weekly variation, with Monday encoded as 0.
- `is_weekend`: Distinguishes weekends from weekdays.
- `month`: Represents seasonal or monthly differences within the available data.

All 31,327 forecasting examples contain valid values for these features, with no missing values.

The engineered features are leakage-safe because they are derived exclusively from `LastUpdated`, which is available at prediction time.

These features will be combined with current and historical occupancy measurements during model training.


## 5. Historical Occupancy Feature Engineering

Historical occupancy measurements provide information about recent parking demand and changes in facility utilization.

We construct lag features using observations from the same parking facility:

- `occupancy_lag_1`: Occupancy at the previous observation.
- `occupancy_lag_2`: Occupancy two observations earlier.
- `occupancy_change_1`: Change in occupancy between the current and previous observation.

To prevent misleading temporal relationships, historical observations must be separated by reasonable reporting intervals.

We require each of the two preceding observation intervals to fall within 25–35 minutes. This avoids treating observations separated by overnight or multi-day gaps as consecutive half-hourly measurements.

Lag features use only past observations and are therefore available at prediction time.


In [18]:
# Sort validated observations chronologically within each facility
df = df.sort_values(
    ["SystemCodeNumber", "LastUpdated"]
).reset_index(drop=True)

groups = df.groupby("SystemCodeNumber")

# Create historical occupancy features
df["occupancy_lag_1"] = groups["Occupancy"].shift(1)
df["occupancy_lag_2"] = groups["Occupancy"].shift(2)

df["occupancy_change_1"] = (
    df["Occupancy"] - df["occupancy_lag_1"]
)

# Calculate rolling mean from current and two previous observations
df["occupancy_rolling_mean_3"] = (
    df.groupby("SystemCodeNumber")["Occupancy"]
    .transform(lambda x: x.rolling(window=3, min_periods=3).mean())
)

# Calculate historical observation intervals
df["lag_interval_1"] = (
    groups["LastUpdated"].diff().dt.total_seconds() / 60
)

df["lag_interval_2"] = (
    df.groupby("SystemCodeNumber")["LastUpdated"]
    .diff(2).dt.total_seconds() / 60
    - df["lag_interval_1"]
)

# Require both historical intervals to be within 25–35 minutes
valid_lags = (
    df["lag_interval_1"].between(25, 35)
    & df["lag_interval_2"].between(25, 35)
)

# Keep lag features only for valid historical sequences
lag_columns = [
    "occupancy_lag_1",
    "occupancy_lag_2",
    "occupancy_change_1",
    "occupancy_rolling_mean_3"
]

df.loc[~valid_lags, lag_columns] = float("nan")

# Join lag features to our existing forecasting examples
df_model = df_model.drop(
    columns=lag_columns,
    errors="ignore"
)

df_model = df_model.merge(
    df[["SystemCodeNumber", "LastUpdated"] + lag_columns],
    on=["SystemCodeNumber", "LastUpdated"],
    how="left",
    validate="one_to_one"
)

# Inspect availability of historical features
print("Total forecasting examples:", len(df_model))
print("Examples with complete lag features:",
      df_model[lag_columns].notna().all(axis=1).sum())

print("\nMissing values in lag features:")
print(df_model[lag_columns].isna().sum())

display(
    df_model[
        ["SystemCodeNumber", "LastUpdated", "Occupancy"]
        + lag_columns
    ].head(10)
)


Total forecasting examples: 31327
Examples with complete lag features: 25793

Missing values in lag features:
occupancy_lag_1             5534
occupancy_lag_2             5534
occupancy_change_1          5534
occupancy_rolling_mean_3    5534
dtype: int64


,SystemCodeNumber,LastUpdated,Occupancy,occupancy_lag_1,occupancy_lag_2,occupancy_change_1,occupancy_rolling_mean_3
0,BHMBCCMKT01,2016-10-04 07:59:42,61,NaN,NaN,NaN,NaN
1,BHMBCCMKT01,2016-10-04 08:25:42,64,NaN,NaN,NaN,NaN
2,BHMBCCMKT01,2016-10-04 08:59:42,80,64.0,61.0,16.0,68.333333
3,BHMBCCMKT01,2016-10-04 09:32:46,107,80.0,64.0,27.0,83.666667
4,BHMBCCMKT01,2016-10-04 09:59:48,150,107.0,80.0,43.0,112.333333
5,BHMBCCMKT01,2016-10-04 10:26:49,177,150.0,107.0,27.0,144.666667
6,BHMBCCMKT01,2016-10-04 10:59:48,219,177.0,150.0,42.0,182.000000
7,BHMBCCMKT01,2016-10-04 11:25:47,247,219.0,177.0,28.0,214.333333
8,BHMBCCMKT01,2016-10-04 11:59:44,259,247.0,219.0,12.0,241.666667
9,BHMBCCMKT01,2016-10-04 12:29:45,266,259.0,247.0,7.0,257.333333


### 5.1 Historical Occupancy Feature Results

Historical occupancy features were constructed using observations from the same parking facility.

The engineered features include:

- `occupancy_lag_1`: Occupancy at the preceding observation.
- `occupancy_lag_2`: Occupancy two observations earlier.
- `occupancy_change_1`: Difference between current and previous occupancy.

To account for irregular timestamps, both historical observation intervals were required to fall within 25–35 minutes.

**Results:**

| Metric | Count |
|---|---:|
| Forecasting examples | 31,327 |
| Examples with complete lag features | 25,793 |
| Examples without complete lag features | 5,534 |
| Complete lag coverage | 82.33% |

**Interpretation:**

Approximately 82.33% of forecasting examples contain two consecutive historical observations within the required time tolerance.

Missing lag features primarily reflect unavailable or temporally unsuitable historical sequences rather than missing values in the original dataset.

For initial model development, examples without complete lag features will be excluded. This provides a consistent feature set for comparing baseline and regression models.

Lag features were computed using historical observations from the validated dataset before joining them to the forecasting examples, preventing artificial gaps caused by target filtering.

All historical occupancy features use information available at or before the prediction timestamp.



### Rolling Occupancy Feature

An additional feature, `occupancy_rolling_mean_3`, represents the average occupancy across the current observation and the two preceding observations for the same parking facility.

This feature captures recent occupancy levels while smoothing short-term fluctuations.

The rolling calculation uses only information available at prediction time and does not incorporate future occupancy.

Like the historical lag features, the rolling mean is considered valid only when both preceding observation intervals fall within 25–35 minutes.

## 6. Modeling Dataset Preparation

The modeling dataset is constructed by retaining forecasting examples with complete historical occupancy features.

Rows with unavailable lag values are excluded rather than imputed because the missing values represent gaps in the historical observation sequence.

The dataset will retain target timestamps for chronological split validation, but these timestamps and future occupancy values will not be included as model inputs.

The resulting dataset will be used for chronological train, validation, and test splitting.


In [19]:
# Retain examples with complete historical features
df_final = df_model.dropna(
    subset=[
        "occupancy_lag_1",
        "occupancy_lag_2",
        "occupancy_change_1",
        "occupancy_rolling_mean_3",
    ]
).copy()

# Sort globally by prediction timestamp
df_final = df_final.sort_values(
    ["LastUpdated", "SystemCodeNumber"]
).reset_index(drop=True)

# Verify the final modeling dataset
print("Final modeling examples:", len(df_final))
print("Parking facilities:", df_final["SystemCodeNumber"].nunique())

print("\nMissing values:")
print(df_final.isna().sum())

print("\nPrediction timestamp range:")
print(df_final["LastUpdated"].min())
print(df_final["LastUpdated"].max())

display(df_final.head())


Final modeling examples: 25793
Parking facilities: 30

Missing values:
SystemCodeNumber            0
Capacity                    0
Occupancy                   0
LastUpdated                 0
TargetTimestamp             0
TargetOccupancy             0
ForecastHorizonMinutes      0
hour                        0
day_of_week                 0
is_weekend                  0
month                       0
occupancy_lag_1             0
occupancy_lag_2             0
occupancy_change_1          0
occupancy_rolling_mean_3    0
dtype: int64

Prediction timestamp range:
2016-10-04 08:59:42
2016-12-19 16:03:35


,SystemCodeNumber,Capacity,Occupancy,LastUpdated,TargetTimestamp,TargetOccupancy,ForecastHorizonMinutes,hour,day_of_week,is_weekend,month,occupancy_lag_1,occupancy_lag_2,occupancy_change_1,occupancy_rolling_mean_3
0,BHMBCCMKT01,577,80,2016-10-04 08:59:42,2016-10-04 09:32:46,107.0,33.066667,8,1,0,10,64.0,61.0,16.0,68.333333
1,BHMBCCPST01,317,109,2016-10-04 08:59:42,2016-10-04 09:32:46,116.0,33.066667,8,1,0,10,98.0,93.0,11.0,100.000000
2,BHMBCCSNH01,863,373,2016-10-04 08:59:42,2016-10-04 09:32:46,503.0,33.066667,8,1,0,10,253.0,177.0,120.0,267.666667
3,BHMBCCTHL01,387,155,2016-10-04 08:59:42,2016-10-04 09:32:46,204.0,33.066667,8,1,0,10,129.0,120.0,26.0,134.666667
4,BHMBRCBRG01,1010,116,2016-10-04 08:59:42,2016-10-04 09:32:46,145.0,33.066667,8,1,0,10,88.0,70.0,28.0,91.333333


### 6.1 Final Modeling Dataset Summary

After excluding observations without complete historical occupancy features, the modeling dataset contains **25,793 examples across 30 parking facilities**.

No missing values remain in the modeling dataset.

The prediction timestamps range from October 4 to December 19, 2016.

The dataset includes current occupancy, historical occupancy measurements, calendar features, facility identifiers, and future occupancy targets.

## 7. Chronological Train/Validation/Test Splitting

To evaluate forecasting performance realistically, observations are divided chronologically into training, validation, and test sets.

The approximate split proportions are:

- Training: 70%
- Validation: 15%
- Test: 15%

Global timestamp boundaries are used across all facilities to avoid training on later observations from one facility while evaluating earlier observations from another.

To prevent temporal leakage, training target timestamps must precede the validation period, and validation target timestamps must precede the test period.

The held-out test set will remain isolated during model selection and hyperparameter tuning. It will only be used for final evaluation and deployment validation.


In [20]:
# Identify chronological boundaries using unique prediction timestamps
unique_times = df_final["LastUpdated"].sort_values().unique()

train_boundary = pd.Timestamp(
    unique_times[int(len(unique_times) * 0.70)]
)

test_boundary = pd.Timestamp(
    unique_times[int(len(unique_times) * 0.85)]
)

# Split using global timestamps
train_df = df_final.loc[
    (df_final["LastUpdated"] < train_boundary)
    & (df_final["TargetTimestamp"] < train_boundary)
].copy()

val_df = df_final.loc[
    (df_final["LastUpdated"] >= train_boundary)
    & (df_final["LastUpdated"] < test_boundary)
    & (df_final["TargetTimestamp"] < test_boundary)
].copy()

test_df = df_final.loc[
    df_final["LastUpdated"] >= test_boundary
].copy()

# Verify temporal separation
assert train_df["TargetTimestamp"].max() < val_df["LastUpdated"].min()
assert val_df["TargetTimestamp"].max() < test_df["LastUpdated"].min()

print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))
print("Test examples:", len(test_df))

print("\nTraining period:")
print(train_df["LastUpdated"].min(), "to", train_df["LastUpdated"].max())

print("\nValidation period:")
print(val_df["LastUpdated"].min(), "to", val_df["LastUpdated"].max())

print("\nTest period:")
print(test_df["LastUpdated"].min(), "to", test_df["LastUpdated"].max())

print("\nTotal examples retained:",
      len(train_df) + len(val_df) + len(test_df))

print("Temporal leakage checks: PASSED")


Training examples: 18423
Validation examples: 3509
Test examples: 3833

Training period:
2016-10-04 08:59:42 to 2016-11-25 16:00:59

Validation period:
2016-11-26 09:01:01 to 2016-12-08 09:59:28

Test period:
2016-12-08 10:59:25 to 2016-12-19 16:03:35

Total examples retained: 25765
Temporal leakage checks: PASSED


### 7.1 Chronological Splitting Results

The final modeling dataset contained 25,793 forecasting examples before chronological splitting.

After enforcing temporal boundaries, 25,765 examples were retained.

| Dataset | Examples | Percentage |
|---|---:|---:|
| Training | 18,423 | 71.5% |
| Validation | 3,509 | 13.6% |
| Test | 3,833 | 14.9% |
| **Total** | **25,765** | **100%** |

**Temporal coverage:**

- Training: October 4 – November 25, 2016
- Validation: November 26 – December 8, 2016
- Test: December 8 – December 19, 2016

A total of 28 examples were excluded because their target timestamps crossed chronological split boundaries.

**Leakage prevention:**

- All splits use global chronological boundaries.
- Training target timestamps precede validation prediction timestamps.
- Validation target timestamps precede test prediction timestamps.
- Historical occupancy features use observations available at or before prediction time.
- Future occupancy targets are excluded from model inputs.
- The test set is reserved for final evaluation and will not be used for model selection.

Both temporal leakage assertions passed successfully.

The resulting datasets are ready for feature selection and model development.


## 8. Model Input Features and Target

The supervised learning task predicts future parking occupancy using information available at the current observation timestamp.

The target variable is `TargetOccupancy`.

**Model input features:**

- `SystemCodeNumber`: Parking facility identifier
- `Capacity`: Total parking capacity
- `Occupancy`: Current occupancy
- `hour`: Hour of observation
- `day_of_week`: Day of week
- `is_weekend`: Weekend indicator
- `month`: Calendar month
- `occupancy_lag_1`: Previous occupancy
- `occupancy_lag_2`: Occupancy two observations earlier
- `occupancy_change_1`: Change in occupancy since the previous observation

**Excluded columns:**

- `LastUpdated`: Original timestamp, already represented by calendar features
- `TargetTimestamp`: Future observation timestamp
- `ForecastHorizonMinutes`: Actual elapsed time to the future observation
- `TargetOccupancy`: Prediction target

The facility identifier will require categorical encoding for regression models.

Any preprocessing that learns parameters from data, including categorical encoding, will be fitted using the training set only and then applied to validation and test sets.


In [21]:
# Define model input features
feature_columns = [
    "SystemCodeNumber",
    "Capacity",
    "Occupancy",
    "hour",
    "day_of_week",
    "is_weekend",
    "month",
    "occupancy_lag_1",
    "occupancy_lag_2",
    "occupancy_change_1",
    "occupancy_rolling_mean_3"
]

target_column = "TargetOccupancy"

# Create training and validation matrices
X_train = train_df[feature_columns].copy()
y_train = train_df[target_column].copy()

X_val = val_df[feature_columns].copy()
y_val = val_df[target_column].copy()

# Keep test data isolated until final evaluation
print("Training features:", X_train.shape)
print("Training targets:", y_train.shape)

print("\nValidation features:", X_val.shape)
print("Validation targets:", y_val.shape)

print("\nSelected features:")
print(X_train.columns.tolist())

print("\nMissing training feature values:",
      X_train.isna().sum().sum())

print("Missing validation feature values:",
      X_val.isna().sum().sum())

# Verify future-derived columns are excluded
for forbidden in [
    "TargetTimestamp",
    "TargetOccupancy",
    "ForecastHorizonMinutes"
]:
    assert forbidden not in X_train.columns
    assert forbidden not in X_val.columns

print("\nFeature leakage checks: PASSED")


Training features: (18423, 11)
Training targets: (18423,)

Validation features: (3509, 11)
Validation targets: (3509,)

Selected features:
['SystemCodeNumber', 'Capacity', 'Occupancy', 'hour', 'day_of_week', 'is_weekend', 'month', 'occupancy_lag_1', 'occupancy_lag_2', 'occupancy_change_1', 'occupancy_rolling_mean_3']

Missing training feature values: 0
Missing validation feature values: 0

Feature leakage checks: PASSED


### 8.1 Model Feature Preparation Results

The chronological training and validation datasets were successfully prepared using 11 input features, including the three-observation rolling occupancy mean.

| Dataset | Feature Matrix Shape | Target Shape |
|---|---|---|
| Training | (18,423, 11) | (18,423,) |
| Validation | (3,509, 11) | (3,509,) |

**Validation results:**

- No missing values were detected in training or validation features.
- All 11 selected input features are available at prediction time.
- Future occupancy and target timestamps were excluded from model inputs.
- Training and validation sets remain chronologically separated.
- The held-out test set remains isolated from model development.

**Next steps:**

- Export the finalized 11-feature schema.
- Use reusable preprocessing functions in the automated Airflow pipeline.
- Fit categorical encoding on training data only.
- Compare baseline and regression models using validation MAE and RMSE.